# PVSG, Option A: the full SVG2 pipeline on PVSG videos

**Goal:** run SVG2's dataset-generation pipeline from the raw video (Option A) on a few PVSG videos, so that `pvsg_eval.ipynb` can compare every stage with PVSG's human labels.

**What runs** (`tools/pvsg_pipeline.py`, one video after another, in the background):

| stage | what | change for PVSG (5 fps videos) |
|---|---|---|
| 1 | SAM2 "segment everything" | every **5th** frame = once per second (theirs: every 20th) |
| 2 | SAM2 tracking, online + offline pass | a new-object check at **every** stage-1 frame = once per second (theirs: 3–5 checks per video) |
| 3 | cleanup | none |
| 4 | DAM descriptions | **all** tracked objects (theirs: the 40 largest) |
| 5 | name + attributes (NIM: nemotron lightning) | none (their prompt) |
| 6 | relations (NIM: kimi-k3) | 1 frame/s, up to 40 frames (theirs: max 24) |
| 7 | ours: map names → PVSG's 126 classes, predicates → PVSG's 57, spans → PVSG frame numbers | new |

**Safe to disconnect:** the runs are background processes, and every stage is saved. If the session restarts, run cells 1, 2, 3, 5 and 6 again: finished stages are skipped.

**Time:** SAM2 is the slow part (roughly an hour per 100 frames on a T4; the first video will tell). The default is the **4 shortest** validation videos, 2 per GPU.

In [ ]:
import os, sys, json, subprocess, time, getpass, shutil
REPO = "/kaggle/working/svg2-implementation"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-b", "claude/traser-implementation-guide-xlshit",
                    "https://github.com/kanu2704/svg2-implementation", REPO], check=True)
os.chdir(REPO)
os.environ["HF_HOME"] = "/tmp/hf_cache"
r = subprocess.run("bash setup_kaggle.sh", shell=True, capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
r = subprocess.run('python3 -c "import dam" 2>/dev/null || python3 -m pip install -q --no-deps '
                   '"git+https://github.com/NVlabs/describe-anything.git"; '
                   'python3 -m pip install -q sentencepiece openai huggingface_hub', shell=True, capture_output=True, text=True)
print(r.stdout[-300:], r.stderr[-300:])
sys.path.insert(0, os.path.join(REPO, "tools"))
import pvsg_data
PVSG_ROOT = "/kaggle/tmp/pvsg" if os.path.isdir("/kaggle/tmp") else "/tmp/pvsg"     # big data: not in the repo
RUNS = "/kaggle/working/pvsg_runs"                                                  # pipeline outputs: not in the repo
os.makedirs(PVSG_ROOT, exist_ok=True); os.makedirs(RUNS, exist_ok=True)
print("PVSG data →", PVSG_ROOT, "| runs →", RUNS)

## 1. Keys

- **NVIDIA NIM key** (stages 5–7): typed into a hidden box once, stored in `/root/.svg2_keys` **outside the repo**, so the background runs can read it. Never type it into a cell.
- **Hugging Face token:** only needed if the PVSG download says you must log in or accept terms (then accept them on https://huggingface.co/datasets/Jingkang/PVSG and set `ASK_HF_TOKEN = True`).

In [ ]:
KEYS = "/root/.svg2_keys"
if not (os.path.exists(KEYS) and "NVIDIA_API_KEY=" in open(KEYS).read()):
    key = getpass.getpass("NVIDIA_API_KEY (hidden): ")
    with open(KEYS, "a") as f:
        f.write(f"NVIDIA_API_KEY={key}\n")
    os.chmod(KEYS, 0o600); del key
print("NIM key stored in", KEYS)
ASK_HF_TOKEN = False
HF_TOKEN = getpass.getpass("HF token (hidden): ") if ASK_HF_TOKEN else None

## 2. Download PVSG (VidOR part only)

From the Hugging Face mirror of PVSG: `pvsg.json` + `vidor_videos.zip` + `vidor_masks.zip`. VidOR is 289 of the 400 videos; Epic-Kitchens and Ego4D are not needed for a start. Already-downloaded files are skipped.

In [ ]:
t0 = time.time()
paths = pvsg_data.download(PVSG_ROOT, sources=("vidor",), token=HF_TOKEN)
for name, p in paths.items():
    print(f"{name:20} {os.path.getsize(p) / 2**30:6.2f} GB   {p}")
shutil.copy(paths["pvsg.json"], os.path.join(PVSG_ROOT, "pvsg.json"))
anno = pvsg_data.load_anno(PVSG_ROOT)
print(f"done in {time.time() - t0:.0f} s | {len(anno['data'])} videos, {len(anno['objects']['thing'])} thing + "
      f"{len(anno['objects']['stuff'])} stuff classes, {len(anno['relations'])} predicates")
print("stuff classes:", anno["objects"]["stuff"])

## 3. Choose the videos

The shortest VidOR **validation** videos with at least 50 frames (10 s), because SAM2 time grows with the number of frames. To choose your own, set `VIDEOS = [...]`.

In [ ]:
N_VIDEOS = 4
VIDEOS = pvsg_data.choose_videos(anno, source="vidor", split="val", n=N_VIDEOS, min_frames=50)
data = {d["video_id"]: d for d in anno["data"]}
print(f"{'video':18} {'frames':>6} {'sec':>5} {'objects':>7} {'stuff':>5} {'relations':>9}  rough hours on a T4")
for v in VIDEOS:
    d = data[v]; n = d["meta"]["num_frames"]
    est = (n / 5 * 30 + n * 2 * 15) / 3600          # stage 1 ~30 s/frame on 1 in 5 frames + 2 tracking passes ~15 s/frame
    print(f"{v:18} {n:>6} {n / d['meta']['fps']:>5.0f} {len(d['objects']):>7} {sum(not o['is_thing'] for o in d['objects']):>5} "
          f"{len(d['relations']):>9}  ~{est:.1f} (+ DAM/NIM ~0.3)")

## 4. Copy out only these videos and check them

Mask PNGs and mp4 frames must agree with `pvsg.json` (`ok: True`).

In [ ]:
zips = {os.path.basename(p): p for p in paths.values()}
for v, rep in pvsg_data.extract_videos(PVSG_ROOT, zips, anno, VIDEOS).items():
    print(v, rep["mask_pngs"], "mask PNGs |", pvsg_data.check_video(PVSG_ROOT, anno, v))

## 5. Start the runs (background, one queue per GPU)

Each GPU gets its own list of videos and works through it. Logs: `pvsg_runs/worker0.log`, `worker1.log`. Running this cell again does **not** start duplicates while workers are alive.

In [ ]:
running = subprocess.run("pgrep -f [t]ools/pvsg_pipeline.py", shell=True, capture_output=True, text=True).stdout.split()
free = subprocess.run(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
                      capture_output=True, text=True).stdout.split()
print("free MiB per GPU:", free)
if running:
    print("workers already running:", running)
else:
    if any(int(f) < 12000 for f in free):
        print("!! a GPU has < 12 GB free: shut down other notebook kernels first (they keep models on the GPU)")
    n_gpu = max(1, len(free))
    for g in range(n_gpu):
        queue = VIDEOS[g::n_gpu]
        if not queue:
            continue
        cmd = (f"CUDA_VISIBLE_DEVICES={g} HF_HOME=/tmp/hf_cache nohup python3 tools/pvsg_pipeline.py "
               f"--pvsg_root {PVSG_ROOT} --out {RUNS} --videos {' '.join(queue)} >> {RUNS}/worker{g}.log 2>&1 &")
        subprocess.run(cmd, shell=True)
        print(f"GPU {g}: {queue}")

## 6. Progress (run again whenever you like)

In [ ]:
def progress(tail=4):
    for v in VIDEOS:
        p = os.path.join(RUNS, v, "status.json")
        st = json.load(open(p)) if os.path.exists(p) else {"state": "waiting"}
        print(f"{v:18} {st.get('state'):8} stage: {st.get('stage', '-'):18} seconds per stage: {st.get('seconds', {})} "
              f"(updated {st.get('updated', '-')}) {st.get('error', '')}")
    for g in range(2):
        p = os.path.join(RUNS, f"worker{g}.log")
        if os.path.exists(p):
            print(f"\n--- worker{g}.log (last {tail} lines)")
            print("".join(open(p).readlines()[-tail:]))
    alive = subprocess.run("pgrep -f [t]ools/pvsg_pipeline.py", shell=True, capture_output=True, text=True).stdout.split()
    print("workers alive:", len(alive))
progress()

## 7. Or just leave it: watch automatically, then evaluate

This cell refreshes the progress every 5 minutes by itself. When every video is `done` (or `failed`) and no worker is left, it runs `pvsg_eval.ipynb` automatically and prints the result. Leave it running; stopping it does **not** stop the pipeline.

In [ ]:
from IPython.display import clear_output
REFRESH_MIN = 5
while True:
    clear_output(wait=True)
    print(time.strftime("%H:%M:%S"), "(refreshes every", REFRESH_MIN, "min)\n")
    progress()
    states = []
    for v in VIDEOS:
        p = os.path.join(RUNS, v, "status.json")
        states.append(json.load(open(p)).get("state") if os.path.exists(p) else "waiting")
    alive = subprocess.run("pgrep -f [t]ools/pvsg_pipeline.py", shell=True, capture_output=True, text=True).stdout.split()
    if not alive and all(s in ("done", "failed") for s in states):
        break
    time.sleep(REFRESH_MIN * 60)

print("\nall runs finished:", dict(zip(VIDEOS, states)))
if "done" in states:
    print("running notebooks/pvsg_eval.ipynb ...")
    r = subprocess.run("jupyter nbconvert --to notebook --execute --inplace "
                       "--ExecutePreprocessor.timeout=-1 notebooks/pvsg_eval.ipynb", shell=True, capture_output=True, text=True)
    print(r.stderr[-1500:])
    if os.path.exists("notes/pvsg/README.md"):
        print(open("notes/pvsg/README.md").read())
        print("\nto save:  git add -f notes/pvsg data/pvsg notebooks/pvsg_eval.ipynb && git commit -m 'pvsg eval' && git push")

**When all videos say `done`:** open `notebooks/pvsg_eval.ipynb` (no GPU needed). Not needed if section 7 ran it for you.

**To stop the runs:** `!pkill -f tools/pvsg_pipeline.py` (finished stages stay saved).